# Modelado y selección del predictor de género

Este notebook elige un clasificador para `main_genre`. No repite el EDA ni la preparación: usa el corte, los transformadores y las exclusiones del notebook 02.

La entrada de cada predicción es el string `chords` de una pista. La etiqueta es el género del artista en Spotify, copiado a esa pista. El artista no entra como variable. Una predicción responde qué género de catálogo corresponde a una progresión cuyo artista no estuvo en el entrenamiento.

Las celdas están escritas para ejecutarse en orden desde un kernel reiniciado. Aquí no hay salidas guardadas.


## 1. Objetivo, métrica y protocolo

**Tipo.** Clasificación de 12 clases: alternative, country, electronic, jazz, metal, pop, pop rock, punk, rap, reggae, rock y soul.

**Métrica principal.** Macro-F1. Promedia el F1 de las doce clases, así que equivocar reggae pesa lo mismo que equivocar pop. Accuracy no decide: pop es cerca del 24 % de las filas y un clasificador constante en pop ya rondaría ese valor con un macro-F1 cercano a cero.

**Métricas secundarias.** Accuracy, para comparar contra el piso de la clase mayoritaria. F1 por clase y matriz de confusión, solo en la evaluación final, para ver si los errores caen dentro de pop, rock, pop rock, alternative y punk. Esas cinco familias comparten lenguaje armónico en el EDA. No se usan para elegir el modelo.

**Corte externo.** El mismo que el notebook 02: filas con `main_genre`, `GroupShuffleSplit` por `artist_id`, 20 % de los artistas, semilla 42. Un artista viaja entero a un solo lado. Test se describe al cortar y no vuelve a usarse hasta la última evaluación.

**Validación.** `StratifiedGroupKFold` de 3 folds, con barajado y semilla 42, solo sobre train. Es el equivalente agrupado que pide la asignación: varias filas por artista, y además conserva las clases raras en cada fold. Hace falta que cada artista tenga una sola etiqueta y que cada clase tenga al menos tres artistas. Esos dos supuestos se comprueban antes de ajustar nada.

Tres folds, no cinco. En el notebook 02, ajustar una vez los ocho vocabularios llevó unos 22 minutos. Cada fold vuelve a aprender el vocabulario solo con sus artistas. Cinco folds casi duplicarían ese costo antes de comparar modelos.

**Misma partición.** Los índices se materializan una vez. Representaciones, clasificadores y trials de Optuna leen esos índices.

**Fuga.** El vocabulario, el imputador y el escalador se ajustan dentro de cada fold. `min_df` 5 y el tope de 5.000 se mantienen fijos: moverlos obligaría a rehacer el vocabulario en cada trial. El test no entra en esa elección, ni en la del modelo, ni en la de los hiperparámetros.


In [ ]:
from __future__ import annotations

import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
TEST_SIZE = 0.20
YEAR_MIN = 1899
YEAR_MAX = 2023
NGRAM_ORDERS = (1, 2, 3, 4)
NGRAM_MIN_DF = 5
NGRAM_MAX_FEATURES = 5000

REQUIRED_COLUMNS = [
    "id",
    "chords",
    "release_date",
    "genres",
    "decade",
    "rock_genre",
    "artist_id",
    "main_genre",
    "spotify_song_id",
    "spotify_artist_id",
]

MAIN_GENRES = frozenset(
    {
        "pop",
        "rock",
        "country",
        "alternative",
        "pop rock",
        "punk",
        "metal",
        "rap",
        "soul",
        "jazz",
        "reggae",
        "electronic",
    }
)

# Identidad, fuente de la etiqueta, otra tarea, tiempo y el target.
EXCLUDED_COLUMNS = [
    "id",
    "artist_id",
    "spotify_song_id",
    "spotify_artist_id",
    "genres",
    "rock_genre",
    "release_date",
    "decade",
    "main_genre",
    "chords",
]

if (Path("data") / "chordonomicon_v2_part_0.csv").exists():
    DATA_DIR = Path("data")
elif (Path("..") / "data" / "chordonomicon_v2_part_0.csv").exists():
    DATA_DIR = Path("..") / "data"
else:
    raise FileNotFoundError(
        "No están los CSV chordonomicon_v2_part_*.csv en data/ ni en ../data/."
    )

print("DATA_DIR =", DATA_DIR.resolve())
print("semilla =", SEED, "| fracción de artistas en test =", TEST_SIZE)
print(
    "n-gramas: órdenes",
    NGRAM_ORDERS,
    "| min_df =",
    NGRAM_MIN_DF,
    "| tope =",
    NGRAM_MAX_FEATURES,
)


In [ ]:
import gc
import time

import matplotlib.pyplot as plt
import optuna
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_SPLITS = 3
N_TRIALS = 20
LABELS = sorted(MAIN_GENRES)

PARAMS_NB = {"alpha": 1.0}
PARAMS_SGD = {"alpha": 1e-4, "penalty": "l2", "class_weight": "balanced"}
PARAMS_SVC = {"C": 1.0}

CONFIG_NB = "alpha=1.0"
CONFIG_SGD = "log_loss, penalty=l2, alpha=1e-4, class_weight=balanced"
CONFIG_SVC = "C=1.0, class_weight=balanced, dual=False, max_iter=1000"
CONFIG_DUMMY = "strategy=most_frequent"

print("Clases:", LABELS)
print("Folds:", N_SPLITS, "| trials de Optuna:", N_TRIALS)


## 2. Carga y corte

Los siete CSV se concatenan y se validan como en el notebook 02. El corte de almacenamiento no es el corte de modelado. Después se aparta el 20 % de los artistas. La tabla de cuotas describe el corte; no modifica el pipeline.


In [ ]:
def load_shards(data_dir: Path) -> pd.DataFrame:
    paths = sorted(data_dir.glob("chordonomicon_v2_part_*.csv"))
    if not paths:
        raise FileNotFoundError(
            f"No hay chordonomicon_v2_part_*.csv en {data_dir.resolve()}."
        )
    frames: list[pd.DataFrame] = []
    header: list[str] | None = None
    for path in paths:
        # Un solo pase de tipos. Con el lector por trozos, un bloque de nulos
        # se infiere como número y el siguiente como texto.
        part = pd.read_csv(path, low_memory=False)
        columns = list(part.columns)
        if header is None:
            header = columns
        elif columns != header:
            raise ValueError(
                f"{path.name} no tiene el mismo encabezado que {paths[0].name}: {columns}"
            )
        frames.append(part)
        print(f"{path.name}: {len(part):,} filas")
    return pd.concat(frames, ignore_index=True)


def validate_raw(df: pd.DataFrame) -> None:
    missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    duplicated_ids = int(df["id"].duplicated().sum())
    if duplicated_ids:
        raise ValueError(f"Hay {duplicated_ids:,} id duplicados. Cada fila debe ser una pista.")

    if df["chords"].isna().any():
        raise ValueError(
            "chords tiene valores ausentes. En el EDA esa columna está completa y es la entrada del modelo."
        )

    present = set(df["main_genre"].dropna().unique())
    unexpected = sorted(present - MAIN_GENRES)
    if unexpected:
        raise ValueError(f"main_genre tiene clases fuera de las 12 esperadas: {unexpected}")
    absent = sorted(MAIN_GENRES - present)
    if absent:
        raise ValueError(f"main_genre no contiene estas clases esperadas: {absent}")

    dated = df["release_date"].notna()
    years = pd.to_numeric(
        df.loc[dated, "release_date"].astype("string").str.slice(0, 4),
        errors="coerce",
    )
    unreadable = int(years.isna().sum())
    if unreadable:
        raise ValueError(
            f"{unreadable:,} release_date no empiezan por un año de cuatro dígitos."
        )
    out_of_range = (years < YEAR_MIN) | (years > YEAR_MAX)
    n_out = int(out_of_range.sum())
    if n_out:
        sample = years.loc[out_of_range].head(5).tolist()
        raise ValueError(
            f"{n_out:,} fechas tienen año fuera de {YEAR_MIN}–{YEAR_MAX}. Ejemplos: {sample}"
        )

    numeric = df.select_dtypes(include="number")
    n_inf = int(np.isinf(numeric.to_numpy(dtype=float)).sum())
    if n_inf:
        raise ValueError(f"Hay {n_inf:,} valores infinitos en columnas numéricas.")

    print("Validación superada.")
    print(f"Filas: {len(df):,} | columnas: {list(df.columns)}")
    print(f"id únicos: {df['id'].nunique():,}")
    print(f"Con main_genre: {int(df['main_genre'].notna().sum()):,}")
    print(f"Años fechados dentro de {YEAR_MIN}–{YEAR_MAX}: {int(dated.sum()):,}")


raw = load_shards(DATA_DIR)
validate_raw(raw)


El split agrupa por artista. No se estratifica por fila: partir un artista pondría su etiqueta en los dos lados. Las cuotas de test pueden diferir de las de train porque un artista entero viaja junto. Esa descripción no se consulta para elegir representación, modelo ni hiperparámetros.


In [ ]:
# USA_TEST: descripcion del corte. No elige modelo.
supervised = raw.loc[raw["main_genre"].notna()].copy()
missing_artist = int(supervised["artist_id"].isna().sum())
if missing_artist:
    raise ValueError(
        f"{missing_artist:,} filas tienen main_genre y no tienen artist_id. "
        "El split agrupado no puede asignarlas sin romper el grupo."
    )

print(f"Universo supervisado: {len(supervised):,} filas, {supervised['artist_id'].nunique():,} artistas")

features = supervised.drop(columns=["main_genre"])
target = supervised["main_genre"]
groups = supervised["artist_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(features, target, groups))

X_train = features.iloc[train_idx].copy()
X_test = features.iloc[test_idx].copy()
y_train = target.iloc[train_idx].copy()
y_test = target.iloc[test_idx].copy()

shared_artists = set(X_train["artist_id"]).intersection(X_test["artist_id"])
shared_ids = set(X_train["id"]).intersection(X_test["id"])
if shared_artists:
    raise ValueError(f"El split comparte {len(shared_artists):,} artist_id entre train y test.")
if shared_ids:
    raise ValueError(f"El split comparte {len(shared_ids):,} id entre train y test.")

print(f"Train: {len(X_train):,} filas | {X_train['artist_id'].nunique():,} artistas")
print(f"Test:  {len(X_test):,} filas | {X_test['artist_id'].nunique():,} artistas")
print("Artistas en común: 0 | id en común: 0")

class_counts = pd.DataFrame(
    {
        "train": y_train.value_counts(),
        "test": y_test.value_counts(),
    }
).fillna(0).astype(int)
class_counts["train_share"] = class_counts["train"] / class_counts["train"].sum()
class_counts["test_share"] = class_counts["test"] / class_counts["test"].sum()
class_counts = class_counts.sort_values("train", ascending=False)
class_counts


## 3. Pipeline del notebook 02

Los transformadores se copian aquí para que este notebook arranque solo, desde un kernel reiniciado. No se recuperan las instancias ya ajustadas del notebook 02: aquellas terminan en `passthrough` y vieron todo train, así que no sirven para comparar folds.

Hay dos familias, y no se concatenan.

El agregado produce 13 columnas. Doce van a imputación por mediana y `StandardScaler`. `has_structure` se imputa con la moda y se queda en 0/1. `n_chords` y `log1p_n_chords` conviven; las proporciones de calidad suman uno. La regularización L2 de los lineales absorbe esa colinealidad. No se quita una columna fuera de la validación.

Los n-gramas son conteos dispersos, en grafía absoluta o en grados, de orden 1 a 4. No se escalan: el escalado densificaría la matriz. El vocabulario de cada fold sale solo de los artistas de ese fold.

`ComplementNB` solo entra en los n-gramas. El agregado escalado tiene negativos, y este modelo exige conteos no negativos. `SGDClassifier` y `LinearSVC` sí se miden también sobre las 13 columnas, como control de esa representación.


In [ ]:
# Copiado del notebook 02, sección 7. Esta celda solo define el
# transformador. No se ajusta con train ni con test.

PART_RE = re.compile(r"^<([A-Za-z]+)_(\d+)>$")
POWER_RE = re.compile(r"^[A-G]s?(?:no3|5)(?:d)?$", re.I)
_SUFFIX_CHUNKS = (
    "minmaj",
    "augmaj",
    "min7b5",
    "m7b5",
    "hmin",
    "hdim",
    "min",
    "maj",
    "dim",
    "aug",
    "sus",
    "add",
    "no3",
    "13",
    "11",
    "9",
    "7",
    "6",
    "5",
    "4",
    "2",
    "b9",
    "b7",
    "b",
    "s",
    "d",
)

# El medio disminuido se reconoce, como en el EDA, y no se emite:
# allí esas grafías no aparecen, así que la columna sería constante.
QUALITY_COLUMNS = (
    "share_major",
    "share_minor",
    "share_power",
    "share_suspended",
    "share_diminished",
    "share_augmented",
    "share_unknown",
)
_QUALITY_INDEX = {
    "major": 0,
    "minor": 1,
    "power": 2,
    "suspended": 3,
    "diminished": 4,
    "augmented": 5,
    "unknown": 6,
}

NUMERIC_FEATURES = [
    "n_chords",
    "log1p_n_chords",
    "n_unique_chords",
    "unique_ratio",
    *QUALITY_COLUMNS,
    "abab_share",
]
BINARY_FEATURES = ["has_structure"]
FEATURE_COLUMNS = NUMERIC_FEATURES + BINARY_FEATURES


def _recognized_suffix(rest: str) -> bool:
    # Misma lectura que el EDA: un resto no reconocido no se llama mayor.
    text = rest.lower()
    while text:
        for chunk in _SUFFIX_CHUNKS:
            if text.startswith(chunk):
                text = text[len(chunk) :]
                break
        else:
            return False
    return True


def chord_quality(token: str) -> str:
    # Raíz A–G, accidente s/b (sus no es sostenido). El slash usa el acorde a la izquierda.
    core = token.split("/", 1)[0]
    if not core or core[0] not in "ABCDEFG":
        return "unknown"
    rest = core[1:]
    if rest and rest[0] in "bB":
        rest = rest[1:]
    elif rest and rest[0] in "sS" and not rest[1:].lower().startswith("us"):
        rest = rest[1:]
    if not _recognized_suffix(rest):
        return "unknown"
    lowered = core.lower()
    if "sus" in lowered:
        return "suspended"
    if "aug" in lowered:
        return "augmented"
    if "hmin" in lowered or "hdim" in lowered or "min7b5" in lowered or "m7b5" in lowered:
        return "half-diminished"
    if "dim" in lowered:
        return "diminished"
    if "no3" in lowered or POWER_RE.match(core):
        return "power"
    if "min" in lowered:
        return "minor"
    return "major"


def _chords_missing(raw) -> bool:
    if raw is None:
        return True
    missing = pd.isna(raw)
    if isinstance(missing, np.ndarray):
        return False
    return bool(missing)


def _require_chords_frame(X, name: str) -> None:
    if not isinstance(X, pd.DataFrame) or "chords" not in X.columns:
        raise ValueError(f"{name} necesita un DataFrame con la columna 'chords'.")


def chord_tokens(raw) -> tuple[list[str], float]:
    """Acordes y bandera de forma. Un nulo no se lee como el texto 'nan'."""
    if _chords_missing(raw):
        raise ValueError(
            "chords está ausente. El transformador no convierte un valor nulo en el texto 'nan'."
        )
    tokens: list[str] = []
    has_structure = 0.0
    for token in str(raw).split():
        if PART_RE.match(token):
            has_structure = 1.0
        else:
            tokens.append(token)
    return tokens, has_structure


class ChordFeaturesTransformer(BaseEstimator, TransformerMixin):
    """Features por pista a partir de `chords`. No estima nada en `fit`."""

    def fit(self, X, y=None):
        _require_chords_frame(X, "ChordFeaturesTransformer")
        for raw in X["chords"].to_numpy():
            chord_tokens(raw)
        self.n_features_in_ = X.shape[1]
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        _require_chords_frame(X, "ChordFeaturesTransformer")
        chords = X["chords"].to_numpy()
        n_rows = len(chords)
        out = np.zeros((n_rows, len(FEATURE_COLUMNS)), dtype=np.float64)
        i_n = FEATURE_COLUMNS.index("n_chords")
        i_log = FEATURE_COLUMNS.index("log1p_n_chords")
        i_unique = FEATURE_COLUMNS.index("n_unique_chords")
        i_ratio = FEATURE_COLUMNS.index("unique_ratio")
        i_abab = FEATURE_COLUMNS.index("abab_share")
        i_structure = FEATURE_COLUMNS.index("has_structure")
        i_quality = FEATURE_COLUMNS.index("share_major")

        for row, raw in enumerate(chords):
            tokens, has_structure = chord_tokens(raw)
            n_tokens = len(tokens)
            out[row, i_n] = n_tokens
            out[row, i_log] = np.log1p(n_tokens)
            out[row, i_unique] = len(set(tokens))
            out[row, i_ratio] = (len(set(tokens)) / n_tokens) if n_tokens else 0.0
            out[row, i_structure] = has_structure
            if n_tokens:
                counts = np.zeros(len(QUALITY_COLUMNS), dtype=np.float64)
                for token in tokens:
                    quality = _QUALITY_INDEX.get(chord_quality(token))
                    if quality is not None:
                        counts[quality] += 1.0
                out[row, i_quality : i_quality + len(QUALITY_COLUMNS)] = counts / n_tokens
            if n_tokens >= 4:
                windows = n_tokens - 3
                abab = 0
                for start in range(windows):
                    if (
                        tokens[start] == tokens[start + 2]
                        and tokens[start + 1] == tokens[start + 3]
                    ):
                        abab += 1
                out[row, i_abab] = abab / windows
        return pd.DataFrame(out, index=X.index, columns=FEATURE_COLUMNS)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(FEATURE_COLUMNS, dtype=object)


In [ ]:
# Copiado del notebook 02, sección 8. El vocabulario se aprende en fit
# y esta celda no llama a fit.

# Krumhansl & Kessler (1982), perfiles en do mayor y do menor.
# La misma rotación que en la sección 10 del EDA.
KS_MAJOR = np.array(
    [6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88]
)
KS_MINOR = np.array(
    [6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17]
)
_KS_TEMPLATES = np.stack(
    [np.roll(KS_MAJOR, shift) for shift in range(12)]
    + [np.roll(KS_MINOR, shift) for shift in range(12)]
)
_KS_CENTERED = _KS_TEMPLATES - _KS_TEMPLATES.mean(axis=1, keepdims=True)
_KS_NORMS = np.linalg.norm(_KS_CENTERED, axis=1)

_PC = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}
# Semitonos desde la tónica hacia los grados 1-7. Los cinco intervalos
# cromáticos se pliegan sobre el grado vecino, como en el EDA.
_FOLD = (1, 2, 2, 3, 3, 4, 5, 5, 6, 6, 7, 7)


def root_pc(token: str) -> int | None:
    """Clase de altura de la tríada a la izquierda de `/`. None si la fundamental no es A–G."""
    core = token.split("/", 1)[0]
    if not core or core[0] not in _PC:
        return None
    rest = core[1:]
    acc = 0
    # Un "G" a secas tiene resto vacío y sigue siendo G; solo un accidente real lo mueve.
    if rest and rest[0] in "bB":
        acc = -1
    elif rest and rest[0] in "sS" and not rest[1:].lower().startswith("us"):
        acc = 1
    return (_PC[core[0]] + acc) % 12


def degree_tokens(tokens: list[str]) -> list[str]:
    """Grado y cualidad respecto de la tonalidad estimada en esta misma pista."""
    if not tokens:
        return []
    hist = np.zeros(12, dtype=np.float64)
    pcs: list[int] = []
    qualities: list[str] = []
    for token in tokens:
        pc = root_pc(token)
        if pc is None:
            pcs.append(-1)
            qualities.append("other")
            continue
        hist[pc] += 1.0
        pcs.append(pc)
        qualities.append(chord_quality(token))
    if hist.sum() == 0:
        return ["other"] * len(tokens)
    centered = hist - hist.mean()
    norm = float(np.linalg.norm(centered))
    if norm == 0.0:
        return ["other"] * len(tokens)
    correlation = (centered @ _KS_CENTERED.T) / (norm * _KS_NORMS)
    tonic = int(np.argmax(correlation)) % 12
    degrees: list[str] = []
    for pc, quality in zip(pcs, qualities):
        if pc < 0:
            degrees.append("other")
        else:
            degree = _FOLD[(pc - tonic) % 12]
            degrees.append(f"{degree}:{quality}")
    return degrees


def _ngram_key(gram: tuple[str, ...]) -> str:
    return " ".join(gram)


class ChordNgramTransformer(BaseEstimator, TransformerMixin):
    """Conteos de n-gramas. El vocabulario se aprende en `fit` y no crece en `transform`."""

    def __init__(self, n=2, alphabet="absolute", min_df=5, max_features=5000):
        self.n = n
        self.alphabet = alphabet
        self.min_df = min_df
        self.max_features = max_features

    def _check_params(self) -> None:
        if not isinstance(self.n, int) or isinstance(self.n, bool) or self.n < 1:
            raise ValueError("n debe ser un entero >= 1.")
        if self.alphabet not in ("absolute", "degree"):
            raise ValueError(
                f"alphabet debe ser 'absolute' o 'degree', no {self.alphabet!r}."
            )
        if not isinstance(self.min_df, int) or isinstance(self.min_df, bool) or self.min_df < 1:
            raise ValueError("min_df debe ser un entero >= 1.")
        if self.max_features is not None and (
            not isinstance(self.max_features, int)
            or isinstance(self.max_features, bool)
            or self.max_features < 1
        ):
            raise ValueError("max_features debe ser un entero >= 1.")

    def _sequence(self, raw) -> list[str]:
        tokens, _ = chord_tokens(raw)
        if self.alphabet == "absolute":
            return tokens
        return degree_tokens(tokens)

    def _ngrams(self, tokens: list[str]) -> list[tuple[str, ...]]:
        n = self.n
        if len(tokens) < n:
            return []
        return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]

    def fit(self, X, y=None):
        _require_chords_frame(X, "ChordNgramTransformer")
        self._check_params()
        document_frequency: Counter[tuple[str, ...]] = Counter()
        for raw in X["chords"].to_numpy():
            document_frequency.update(set(self._ngrams(self._sequence(raw))))
        ranked = sorted(
            (
                (gram, frequency)
                for gram, frequency in document_frequency.items()
                if frequency >= self.min_df
            ),
            key=lambda item: (-item[1], _ngram_key(item[0])),
        )
        if self.max_features is not None:
            ranked = ranked[: self.max_features]
        self.vocabulary_ = {
            _ngram_key(gram): index for index, (gram, _) in enumerate(ranked)
        }
        self.n_features_in_ = X.shape[1]
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        _require_chords_frame(X, "ChordNgramTransformer")
        self._check_params()
        sequences = [self._sequence(raw) for raw in X["chords"].to_numpy()]
        if not hasattr(self, "vocabulary_"):
            raise RuntimeError(
                "ChordNgramTransformer no está ajustado. Llama a fit antes de transform."
            )
        n_rows = len(sequences)
        n_cols = len(self.vocabulary_)
        vocabulary = self.vocabulary_
        indptr = np.zeros(n_rows + 1, dtype=np.int64)
        indices: list[int] = []
        data: list[float] = []
        for row, tokens in enumerate(sequences):
            counts: dict[int, int] = {}
            for gram in self._ngrams(tokens):
                column = vocabulary.get(_ngram_key(gram))
                if column is not None:
                    counts[column] = counts.get(column, 0) + 1
            for column in sorted(counts):
                indices.append(column)
                data.append(float(counts[column]))
            indptr[row + 1] = len(indices)
        return sparse.csr_matrix(
            (
                np.asarray(data, dtype=np.float64),
                np.asarray(indices, dtype=np.int32),
                indptr,
            ),
            shape=(n_rows, n_cols),
        )

    def get_feature_names_out(self, input_features=None):
        names: list[str | None] = [None] * len(self.vocabulary_)
        for key, index in self.vocabulary_.items():
            names[index] = key
        return np.asarray(names, dtype=object)


## 4. Fábricas sin ajustar

Cada llamada construye objetos nuevos. Un pipeline ajustado no se reutiliza en otro fold ni en otro modelo. La caché de un fold ajusta el transformador solo con las filas de ajuste de ese fold y transforma las de validación. Los clasificadores entrenan después, sobre esas matrices, así el vocabulario no se repite por cada modelo ni por cada trial.

La regla de desempate es la misma en todo el notebook. Gana la media de macro-F1 más alta. Si la diferencia con el siguiente no supera la desviación del líder entre folds, se queda la opción más simple: menos columnas entre representaciones, menos complejidad entre modelos. La complejidad crece en este orden: ComplementNB, LinearSVC, SGD inicial, SGD ajustado.


In [ ]:
def assert_unfitted(estimator) -> None:
    leaked = [
        name
        for name in ("vocabulary_", "mean_", "statistics_", "classes_", "coef_")
        if hasattr(estimator, name)
    ]
    if leaked:
        raise RuntimeError(
            f"{estimator.__class__.__name__} llegó ajustado ({', '.join(leaked)}). "
            "Las fábricas tienen que entregar estimadores sin fit."
        )
    steps = getattr(estimator, "named_steps", None)
    if steps is not None:
        for step in steps.values():
            assert_unfitted(step)
    if estimator.__class__.__name__ == "ColumnTransformer":
        for _, nested, _ in estimator.transformers:
            assert_unfitted(nested)


def make_aggregate_steps() -> list:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    binary_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
        ]
    )
    preprocess = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("binary", binary_pipeline, BINARY_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    return [
        ("chord_features", ChordFeaturesTransformer()),
        ("preprocess", preprocess),
    ]


def make_aggregate_preprocessor() -> Pipeline:
    return Pipeline(steps=make_aggregate_steps())


def make_ngram_transformer(name: str) -> ChordNgramTransformer:
    alphabet, order = name.split("_", 1)
    return ChordNgramTransformer(
        n=int(order),
        alphabet=alphabet,
        min_df=NGRAM_MIN_DF,
        max_features=NGRAM_MAX_FEATURES,
    )


def make_sgd(params: dict) -> SGDClassifier:
    kwargs = {
        "loss": "log_loss",
        "penalty": params["penalty"],
        "alpha": params["alpha"],
        "class_weight": params["class_weight"],
        "random_state": SEED,
        "max_iter": 1000,
        "tol": 1e-3,
    }
    if params["penalty"] == "elasticnet":
        kwargs["l1_ratio"] = params["l1_ratio"]
    return SGDClassifier(**kwargs)


def build_estimator(kind: str, params: dict):
    if kind == "dummy":
        return DummyClassifier(strategy="most_frequent", random_state=SEED)
    if kind == "complement_nb":
        return ComplementNB(alpha=params["alpha"])
    if kind == "sgd":
        return make_sgd(params)
    if kind == "linear_svc":
        return LinearSVC(
            C=params.get("C", 1.0),
            class_weight="balanced",
            dual=False,
            random_state=SEED,
            max_iter=1000,
        )
    raise ValueError(f"Estimador desconocido: {kind}")


def make_pipeline(representation: str, estimator) -> Pipeline:
    if representation == "aggregate":
        return Pipeline(steps=make_aggregate_steps() + [("classifier", estimator)])
    return Pipeline(
        steps=[
            ("ngrams", make_ngram_transformer(representation)),
            ("classifier", estimator),
        ]
    )


def cache_folds(transformer, frame, target, splits):
    assert_unfitted(transformer)
    folds = []
    started = time.perf_counter()
    for train_pos, val_pos in splits:
        step = clone(transformer)
        Z_fit = step.fit_transform(frame.iloc[train_pos])
        Z_val = step.transform(frame.iloc[val_pos])
        if Z_fit.shape[0] != len(train_pos) or Z_val.shape[0] != len(val_pos):
            raise ValueError("El transformador cambió el número de filas.")
        if Z_fit.shape[1] == 0:
            raise ValueError("Un fold produjo cero columnas.")
        folds.append(
            {
                "Z_fit": Z_fit,
                "y_fit": target.iloc[train_pos].to_numpy(),
                "Z_val": Z_val,
                "y_val": target.iloc[val_pos].to_numpy(),
            }
        )
    assert_unfitted(transformer)
    return folds, time.perf_counter() - started


def evaluate_estimator(kind: str, params: dict, folds) -> dict:
    f1_folds = []
    accuracies = []
    fit_times = []
    for fold in folds:
        model = build_estimator(kind, params)
        started = time.perf_counter()
        model.fit(fold["Z_fit"], fold["y_fit"])
        fit_times.append(time.perf_counter() - started)
        predicted = model.predict(fold["Z_val"])
        f1_folds.append(
            f1_score(
                fold["y_val"],
                predicted,
                average="macro",
                labels=LABELS,
                zero_division=0,
            )
        )
        accuracies.append(accuracy_score(fold["y_val"], predicted))
    f1_values = np.asarray(f1_folds, dtype=float)
    return {
        "f1_media": float(f1_values.mean()),
        "f1_desv": float(f1_values.std(ddof=1)),
        "accuracy_media": float(np.mean(accuracies)),
        "tiempo_ajuste_s": float(np.mean(fit_times)),
        "f1_folds": [float(value) for value in f1_values],
    }


def nueva_fila(
    representacion,
    modelo,
    configuracion,
    kind,
    params,
    complejidad,
    n_features,
    tiempo_cache_s,
    metrics,
) -> dict:
    row = {
        "representacion": representacion,
        "modelo": modelo,
        "configuracion": configuracion,
        "kind": kind,
        "params": params,
        "complejidad": complejidad,
        "n_features": n_features,
        "tiempo_cache_s": tiempo_cache_s,
    }
    row.update(metrics)
    return row


def elegir_por_margen(rows: list[dict], simplicity_key: str):
    ranked = sorted(
        rows,
        key=lambda row: (
            -row["f1_media"],
            row[simplicity_key],
            row["f1_desv"],
            row["tiempo_ajuste_s"],
        ),
    )
    best = ranked[0]
    second = ranked[1] if len(ranked) > 1 else ranked[0]
    margin = best["f1_media"] - second["f1_media"]
    if margin <= best["f1_desv"]:
        chosen = min(
            (best, second),
            key=lambda row: (
                row[simplicity_key],
                row["f1_desv"],
                row["tiempo_ajuste_s"],
                -row["f1_media"],
            ),
        )
    else:
        chosen = best
    return best, second, chosen


def explicar_margen(best, second, chosen, name_key: str) -> None:
    margin = best["f1_media"] - second["f1_media"]
    print(
        f"Líder: {best[name_key]} / {best['modelo']} — "
        f"macro-F1 {best['f1_media']:.4f} ± {best['f1_desv']:.4f}"
    )
    print(
        f"Siguiente: {second[name_key]} / {second['modelo']} — "
        f"macro-F1 {second['f1_media']:.4f} ± {second['f1_desv']:.4f}"
    )
    print(f"Diferencia {margin:.4f}. Desviación del líder {best['f1_desv']:.4f}.")
    if margin > best["f1_desv"] and chosen is best:
        print(f"La diferencia supera la desviación. Queda {chosen[name_key]} / {chosen['modelo']}.")
    else:
        print(
            "La diferencia no supera la desviación entre folds. "
            f"Queda la opción más simple: {chosen[name_key]} / {chosen['modelo']}."
        )
    print("Folds del elegido:", [round(value, 4) for value in chosen["f1_folds"]])


def as_table(rows: list[dict], extra=()):
    columns = [
        ("representacion", "Representación"),
        ("modelo", "Modelo"),
        ("configuracion", "Configuración"),
        ("f1_media", "Media"),
        ("f1_desv", "Desv."),
        ("tiempo_ajuste_s", "Tiempo"),
        ("accuracy_media", "Accuracy"),
        ("n_features", "Columnas"),
        *extra,
    ]
    frame = pd.DataFrame(rows)
    show = pd.DataFrame({label: frame[key] for key, label in columns})
    for column in ("Media", "Desv.", "Accuracy"):
        show[column] = show[column].map(lambda value: round(float(value), 4))
    for column in ("Tiempo", "Tiempo caché"):
        if column in show.columns:
            show[column] = show[column].map(lambda value: round(float(value), 1))
    return show.sort_values(["Media", "Desv."], ascending=[False, True]).reset_index(drop=True)


def describe_sgd(params: dict) -> str:
    weight = "None" if params["class_weight"] is None else params["class_weight"]
    text = (
        f"log_loss, penalty={params['penalty']}, "
        f"alpha={params['alpha']:.6g}, class_weight={weight}"
    )
    if params["penalty"] == "elasticnet":
        text += f", l1_ratio={params['l1_ratio']:.3f}"
    return text


print("Fábricas listas. Ningún estimador se ha ajustado.")


## 5. Validación cruzada

Los tres folds se fijan antes de los modelos. El guardia comprueba la etiqueta única por artista y, como mínimo, tres artistas por clase. Después se comprueba que ningún artista aparezca en el ajuste y en la validación del mismo fold.


In [ ]:
artist_labels = (
    pd.DataFrame(
        {
            "artist_id": X_train["artist_id"].to_numpy(),
            "main_genre": y_train.to_numpy(),
        }
    )
    .groupby("artist_id", sort=False)["main_genre"]
    .nunique()
)
mixed = int((artist_labels > 1).sum())
if mixed:
    raise ValueError(
        f"{mixed} artistas de train tienen más de un main_genre. "
        "StratifiedGroupKFold dejaría de estar definido."
    )

artists_per_class = (
    pd.DataFrame(
        {
            "artist_id": X_train["artist_id"].to_numpy(),
            "main_genre": y_train.to_numpy(),
        }
    )
    .drop_duplicates()
    .groupby("main_genre")["artist_id"]
    .nunique()
)
short = artists_per_class[artists_per_class < N_SPLITS]
if len(short):
    raise ValueError(
        f"Estas clases tienen menos de {N_SPLITS} artistas en train: "
        f"{short.to_dict()}."
    )

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
cv_splits = list(cv.split(X_train, y_train, X_train["artist_id"].to_numpy()))
if len(cv_splits) != N_SPLITS:
    raise ValueError(f"Se esperaban {N_SPLITS} folds y salieron {len(cv_splits)}.")

for fold_id, (train_pos, val_pos) in enumerate(cv_splits, start=1):
    if len(set(train_pos).intersection(val_pos)):
        raise ValueError(f"El fold {fold_id} repite filas.")
    train_artists = set(X_train.iloc[train_pos]["artist_id"])
    val_artists = set(X_train.iloc[val_pos]["artist_id"])
    shared = train_artists.intersection(val_artists)
    if shared:
        raise ValueError(f"El fold {fold_id} comparte {len(shared)} artistas.")
    print(
        f"Fold {fold_id}: ajuste {len(train_pos):,} filas, "
        f"{len(train_artists):,} artistas | "
        f"validación {len(val_pos):,} filas, {len(val_artists):,} artistas"
    )

print("Artistas por clase en train:")
print(artists_per_class.sort_values(ascending=False).to_string())
print("Los índices quedan fijos para el resto del notebook.")


## 6. Baseline y candidatos

El piso es `DummyClassifier(strategy="most_frequent")`: siempre predice la clase más frecuente del fold, que en este corte es pop. Ignora los acordes. Su macro-F1 es el número que los demás tienen que superar fuera de la variación entre folds.

Los tres modelos son lineales y distintos. Los tres leen la matriz dispersa de conteos. Ninguno ve `artist_id`.

**ComplementNB.** Estima qué n-gramas son frecuentes en cada género y corrige el sesgo hacia las clases grandes. Necesita conteos no negativos, así que va sobre los n-gramas y no sobre el agregado escalado. Es barato y es el que criba las ocho representaciones. Su límite es la independencia entre n-gramas: una pareja de acordes solo cuenta si el orden del n-grama ya la incluyó. Configuración inicial: `alpha=1.0`.

**SGDClassifier con `loss="log_loss"`.** Es una logística entrenada por lotes, con `class_weight="balanced"`. Cabe en 283 mil filas y 5.000 columnas, y cada trial cambia `alpha`, la penalización y el peso de clase sin rehacer el vocabulario. Se mueve más entre corridas que una logística cerrada; la semilla 42 fija el orden de las filas. Configuración inicial: `penalty="l2"`, `alpha=1e-4`, `class_weight="balanced"`, `max_iter=1000`.

**LinearSVC.** Doce clasificadores de margen, uno por género, en forma primal (`dual=False`) porque hay muchas más filas que columnas. El macro-F1 no necesita probabilidades. No se calibra: la calibración multiplicaría el costo y la matriz de confusión se lee con la clase asignada. Configuración inicial: `C=1.0`, `class_weight="balanced"`, `dual=False`, `max_iter=1000`.

Pasan a Optuna solo el SGD. ComplementNB tiene un único parámetro útil ya fijado en la criba, y LinearSVC se queda en su C inicial como control de margen. El SGD es el candidato cuyo resultado inicial puede deberse a `alpha` y a la penalización. El ajuste se acepta solo si la mejora supera la desviación del SGD inicial.


## 7. Comparación inicial

Primero, ComplementNB en los ocho n-gramas, con los mismos folds. El vocabulario de cada representación se ajusta y se suelta antes de pasar a la siguiente, para no guardar ocho matrices a la vez. La criba elige el n-grama. No elige todavía el clasificador final.

En paralelo, el agregado de 13 columnas se evalúa con el SGD inicial y con LinearSVC. Es el control que el notebook 02 dejó abierto. Esas dos filas no compiten dentro de la criba de n-gramas, porque el modelo no es el mismo. Entran al final, y solo reemplazan al ganador de los n-gramas si lo superan por más que la desviación de ambos.

La columna Tiempo es el ajuste del clasificador. El tiempo de la caché, que es el del vocabulario, se imprime aparte y también queda en la tabla de representaciones.


In [ ]:
screen_rows = []
ngram_names = [
    f"{alphabet}_{order}"
    for alphabet in ("absolute", "degree")
    for order in NGRAM_ORDERS
]
for name in ngram_names:
    print(f"Caché y ComplementNB: {name}")
    folds, cache_s = cache_folds(
        make_ngram_transformer(name), X_train, y_train, cv_splits
    )
    metrics = evaluate_estimator("complement_nb", PARAMS_NB, folds)
    n_features = int(folds[0]["Z_fit"].shape[1])
    screen_rows.append(
        nueva_fila(
            name,
            "ComplementNB",
            CONFIG_NB,
            "complement_nb",
            PARAMS_NB,
            0,
            n_features,
            cache_s,
            metrics,
        )
    )
    print(
        f"{name}: macro-F1 {metrics['f1_media']:.4f} ± {metrics['f1_desv']:.4f}, "
        f"caché {cache_s:.1f} s, columnas {n_features}"
    )
    del folds

print("Caché del agregado")
aggregate_folds, aggregate_cache_s = cache_folds(
    make_aggregate_preprocessor(), X_train, y_train, cv_splits
)
aggregate_n_features = int(aggregate_folds[0]["Z_fit"].shape[1])
if aggregate_n_features != len(FEATURE_COLUMNS):
    raise ValueError(
        f"El agregado produjo {aggregate_n_features} columnas "
        f"y se esperaban {len(FEATURE_COLUMNS)}."
    )
aggregate_specs = [
    ("sgd", "SGDClassifier", CONFIG_SGD, PARAMS_SGD, 2),
    ("linear_svc", "LinearSVC", CONFIG_SVC, PARAMS_SVC, 1),
]
aggregate_rows = []
for kind, modelo, config, params, complejidad in aggregate_specs:
    print(f"Agregado: {modelo}")
    metrics = evaluate_estimator(kind, params, aggregate_folds)
    aggregate_rows.append(
        nueva_fila(
            "aggregate",
            modelo,
            config,
            kind,
            params,
            complejidad,
            aggregate_n_features,
            aggregate_cache_s,
            metrics,
        )
    )
del aggregate_folds
gc.collect()

best_ngram, second_ngram, ngram_choice = elegir_por_margen(screen_rows, "n_features")
explicar_margen(best_ngram, second_ngram, ngram_choice, "representacion")
_, _, aggregate_choice = elegir_por_margen(aggregate_rows, "complejidad")
print(
    "Control agregado:",
    aggregate_choice["modelo"],
    f"{aggregate_choice['f1_media']:.4f} ± {aggregate_choice['f1_desv']:.4f}",
)
print(
    "Los tres clasificadores se comparan en",
    ngram_choice["representacion"],
    ". ComplementNB no admite el agregado escalado.",
)
as_table(
    screen_rows + aggregate_rows,
    extra=(("tiempo_cache_s", "Tiempo caché"),),
)


La celda anterior imprime el líder, el siguiente y la diferencia contra la desviación del líder. Esa frase es la elección del n-grama. Una diferencia menor que la desviación no se lee como superioridad: por eso el vocabulario más chico de esos dos se queda con el puesto.

El tiempo de caché es el costo de reaprender el vocabulario tres veces. El tiempo de la tabla, en la columna Tiempo, es solo el del clasificador. Comparar representaciones por el segundo y olvidar el primero escondería el costo real del orden alto.

## 8. Los tres modelos en el n-grama elegido

Se rehace la caché de esa representación con los mismos índices. Sobre esas matrices entrenan el dummy, ComplementNB, el SGD inicial y LinearSVC. La tabla es la configuración inicial. La selección espera al ajuste del SGD.


In [ ]:
print("Rehaciendo la caché de", ngram_choice["representacion"])
winner_folds, winner_cache_s = cache_folds(
    make_ngram_transformer(ngram_choice["representacion"]),
    X_train,
    y_train,
    cv_splits,
)
winner_features = int(winner_folds[0]["Z_fit"].shape[1])
model_specs = [
    ("dummy", "DummyClassifier", CONFIG_DUMMY, {}, -1),
    ("complement_nb", "ComplementNB", CONFIG_NB, PARAMS_NB, 0),
    ("sgd", "SGDClassifier", CONFIG_SGD, PARAMS_SGD, 2),
    ("linear_svc", "LinearSVC", CONFIG_SVC, PARAMS_SVC, 1),
]
model_rows = []
for kind, modelo, config, params, complejidad in model_specs:
    print("Ajustando", modelo)
    metrics = evaluate_estimator(kind, params, winner_folds)
    model_rows.append(
        nueva_fila(
            ngram_choice["representacion"],
            modelo,
            config,
            kind,
            params,
            complejidad,
            winner_features,
            winner_cache_s,
            metrics,
        )
    )
    print(
        f"{modelo}: macro-F1 {metrics['f1_media']:.4f} ± {metrics['f1_desv']:.4f}, "
        f"ajuste {metrics['tiempo_ajuste_s']:.1f} s"
    )

print(f"Caché compartida por los cuatro: {winner_cache_s:.1f} s")
print("Esta tabla es la configuración inicial. Optuna todavía no ha corrido.")
as_table(model_rows)


## 9. Optuna sobre SGD

La función objetivo es la media de macro-F1 en los tres folds ya cacheados. Se maximiza. El vocabulario no se vuelve a ajustar: los trials solo cambian el clasificador.

Espacio: `alpha` logarítmico entre `1e-6` y `1e-2`; penalización `l2`, `l1` o `elasticnet`; `l1_ratio` entre 0,05 y 0,95 solo si la penalización es elastic net; `class_weight` en `balanced` o ninguno. Presupuesto: 20 trials. Sampler: `TPESampler` con semilla 42. `max_iter` y `tol` se quedan en 1000 y `1e-3`.

`min_df` y el tope de 5.000 no entran al espacio. Cada valor nuevo reharía el vocabulario dentro de cada fold, y ese es el costo que esta máquina no puede multiplicar por 20. Con más recursos, el siguiente ensayo sería mover esos dos cortes solo en el alfabeto y el orden ya elegidos.

Después del estudio se vuelve a evaluar el mejor trial en los mismos folds, para tener media y desviación, y se compara con el SGD inicial. El número suelto del mejor trial no basta.


In [ ]:
def objective(trial: optuna.Trial) -> float:
    penalty = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])
    params = {
        "alpha": trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
        "penalty": penalty,
        "class_weight": {"balanced": "balanced", "none": None}[
            trial.suggest_categorical("class_weight", ["balanced", "none"])
        ],
    }
    if penalty == "elasticnet":
        params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.05, 0.95)
    scores = []
    for fold in winner_folds:
        model = build_estimator("sgd", params)
        model.fit(fold["Z_fit"], fold["y_fit"])
        predicted = model.predict(fold["Z_val"])
        scores.append(
            f1_score(
                fold["y_val"],
                predicted,
                average="macro",
                labels=LABELS,
                zero_division=0,
            )
        )
    return float(np.mean(scores))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study_started = time.perf_counter()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
study_seconds = time.perf_counter() - study_started

best_trial_params = study.best_params
tuned_params = {
    "alpha": best_trial_params["alpha"],
    "penalty": best_trial_params["penalty"],
    "class_weight": None if best_trial_params["class_weight"] == "none" else "balanced",
}
if tuned_params["penalty"] == "elasticnet":
    tuned_params["l1_ratio"] = best_trial_params["l1_ratio"]

tuned_metrics = evaluate_estimator("sgd", tuned_params, winner_folds)
tuned_row = nueva_fila(
    ngram_choice["representacion"],
    "SGDClassifier ajustado",
    describe_sgd(tuned_params),
    "sgd",
    tuned_params,
    3,
    winner_features,
    winner_cache_s,
    tuned_metrics,
)
initial_sgd = next(row for row in model_rows if row["modelo"] == "SGDClassifier")
tuning_gain = tuned_row["f1_media"] - initial_sgd["f1_media"]

print(f"Estudio: {len(study.trials)} trials en {study_seconds:.1f} s")
print(f"Mejor trial: {study.best_value:.4f}")
print(f"Reevaluación: {tuned_row['f1_media']:.4f} ± {tuned_row['f1_desv']:.4f}")
print(f"Parámetros: {tuned_row['configuracion']}")
print(
    f"SGD inicial: {initial_sgd['f1_media']:.4f} ± {initial_sgd['f1_desv']:.4f}. "
    f"Diferencia {tuning_gain:.4f}. Desviación del inicial {initial_sgd['f1_desv']:.4f}."
)
if tuning_gain > initial_sgd["f1_desv"]:
    print("El ajuste supera la variación entre folds del SGD inicial.")
else:
    print("El ajuste no supera la variación entre folds del SGD inicial.")

trial_table = (
    pd.DataFrame(
        [
            {"trial": trial.number, "macro_f1": trial.value, **trial.params}
            for trial in study.trials
            if trial.value is not None
        ]
    )
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
trial_table["macro_f1"] = trial_table["macro_f1"].map(lambda value: round(float(value), 4))
trial_table


La reevaluación manda, no el valor suelto del mejor trial. Si la diferencia contra el SGD inicial no supera la desviación de ese inicial, el ajuste no demostró una mejora. Los dos siguen en la tabla de selección: la regla de margen puede quedarse con el inicial por ser más simple.

## 10. Selección y evaluación única

El pool es ComplementNB, SGD inicial, SGD ajustado y LinearSVC, todos en el n-grama elegido. El dummy se muestra como piso y no se elige. El agregado reemplaza a ese ganador solo si su media lo supera por más que la mayor de las dos desviaciones.

Después de esa decisión se suelta la caché, se construye un pipeline nuevo y se ajusta con todo train. El vocabulario de esa pasada ve todos los artistas de train, que es lo correcto para el modelo final, y no vio test. Hay una sola predicción. Repetir la celda en el mismo kernel se detiene a propósito.


In [ ]:
# USA_TEST: evaluacion final unica. La elección ya está cerrada con los folds.
if globals().get("_test_evaluado", False):
    raise RuntimeError(
        "Test ya fue evaluado en esta sesión del kernel. "
        "Reinicia el kernel para no repetir la predicción."
    )

pool = [row for row in model_rows if row["kind"] != "dummy"] + [tuned_row]
best_model, second_model, chosen_row = elegir_por_margen(pool, "complejidad")
explicar_margen(best_model, second_model, chosen_row, "representacion")

aggregate_margin = aggregate_choice["f1_media"] - chosen_row["f1_media"]
aggregate_threshold = max(chosen_row["f1_desv"], aggregate_choice["f1_desv"])
print(
    f"Agregado {aggregate_choice['modelo']}: "
    f"{aggregate_choice['f1_media']:.4f} ± {aggregate_choice['f1_desv']:.4f}. "
    f"Diferencia {aggregate_margin:.4f}. Umbral {aggregate_threshold:.4f}."
)
if aggregate_margin > aggregate_threshold:
    print("El agregado supera ese umbral. El pipeline final usa las 13 columnas.")
    chosen_row = aggregate_choice
else:
    print("El agregado no supera ese umbral. El pipeline final usa los n-gramas.")

dummy_row = next(row for row in model_rows if row["kind"] == "dummy")
dummy_margin = chosen_row["f1_media"] - dummy_row["f1_media"]
print(
    "El dummy ignora la matriz. Con estos mismos folds su macro-F1 vale "
    "igual para el agregado y para los n-gramas."
)
print(
    f"Dummy: {dummy_row['f1_media']:.4f}. "
    f"Ventaja del elegido {dummy_margin:.4f}. "
    f"Desviación del elegido {chosen_row['f1_desv']:.4f}."
)
if dummy_margin <= chosen_row["f1_desv"]:
    print("Esa ventaja no supera la variación entre folds.")
else:
    print("Esa ventaja supera la variación entre folds.")

winner_folds = None
objective = None
gc.collect()

final_estimator = build_estimator(chosen_row["kind"], chosen_row["params"])
final_pipeline = make_pipeline(chosen_row["representacion"], final_estimator)
assert_unfitted(final_pipeline)

shared_artists = set(X_train["artist_id"]).intersection(set(X_test["artist_id"]))
if shared_artists:
    raise ValueError(f"Train y test comparten {len(shared_artists)} artistas.")

final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)
_test_evaluado = True

test_f1 = f1_score(y_test, y_pred, average="macro", labels=LABELS, zero_division=0)
test_accuracy = accuracy_score(y_test, y_pred)
print(
    f"Pipeline final: {chosen_row['representacion']} + {chosen_row['modelo']}. "
    f"Configuración: {chosen_row['configuracion']}."
)
print(
    f"Validación {chosen_row['f1_media']:.4f} ± {chosen_row['f1_desv']:.4f}. "
    f"Test: macro-F1 {test_f1:.4f}, accuracy {test_accuracy:.4f}."
)
print("Ese número de test no se usa para cambiar la elección.")


## 11. Errores y limitaciones

La matriz y el F1 por clase describen la única predicción ya hecha. No eligen otro modelo.

La pregunta de lectura es cuántos errores se quedan dentro de pop, rock, pop rock, alternative y punk. El EDA midió que ese bloque comparte progresiones. Una fracción alta dentro del bloque es el techo de una etiqueta de artista leída desde una sola progresión. Una fracción alta fuera del bloque, por ejemplo jazz confundido con pop, diría que el modelo no está usando el contraste armónico que el EDA sí veía.

Limitaciones que no dependen del número que salga:

- La etiqueta es el género del catálogo del artista, no un género anotado pista por pista.
- Quedan 110 grupos, 267 filas, con la misma progresión en artistas distintos. El corte por artista no los separa todos.
- El EDA que motivó las features vio también las filas que ahora están en test. Este notebook no usa esas filas para elegir.
- Pop es cerca del 24 % y electronic cerca del 0,8 %. El macro-F1 trata igual esas clases; la matriz dice si las raras se están prediciendo.
- Tres folds dejan una desviación ancha. Una mejora menor que esa desviación no se declara.
- Los tres modelos son lineales. Un árbol sobre las 13 columnas, o un orden de n-grama distinto para cada modelo, queda fuera de este presupuesto.
- `min_df` y el tope de 5.000 no se buscaron.


In [ ]:
# USA_TEST: lectura de la misma predicción. No llama a predict ni a fit.
if "y_pred" not in globals():
    raise RuntimeError("Falta la predicción única. Ejecuta antes la celda de selección.")

report = classification_report(
    y_test,
    y_pred,
    labels=LABELS,
    output_dict=True,
    zero_division=0,
)
per_class = (
    pd.DataFrame(report)
    .T.loc[LABELS, ["precision", "recall", "f1-score", "support"]]
    .sort_values("f1-score")
)
per_class["support"] = per_class["support"].astype(int)
for column in ("precision", "recall", "f1-score"):
    per_class[column] = per_class[column].map(lambda value: round(float(value), 4))

cm = confusion_matrix(y_test, y_pred, labels=LABELS)
block = ("pop", "rock", "pop rock", "alternative", "punk")
inside = 0
total_errors = 0
pairs = []
for i, true_label in enumerate(LABELS):
    for j, pred_label in enumerate(LABELS):
        if i == j:
            continue
        count = int(cm[i, j])
        if count == 0:
            continue
        total_errors += count
        pairs.append((count, true_label, pred_label))
        if true_label in block and pred_label in block:
            inside += count
pairs.sort(reverse=True)

print("F1 por clase, de peor a mejor:")
print(per_class.to_string())
print(f"Errores fuera de la diagonal: {total_errors:,}")
if total_errors:
    print(
        f"Errores dentro de pop / rock / pop rock / alternative / punk: "
        f"{inside:,} ({inside / total_errors:.1%})"
    )
print("Confusiones más frecuentes (real → predicho):")
for count, true_label, pred_label in pairs[:10]:
    print(f"{count:6d}  {true_label} → {pred_label}")

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS).plot(
    ax=ax,
    xticks_rotation=45,
    colorbar=False,
)
ax.set_title("Matriz de confusión de la predicción única")
fig.tight_layout()


## 12. Cierre

La celda de selección imprime qué pipeline quedó, su media y su desviación en validación, si el ajuste de SGD superó la variación del inicial, y el macro-F1 de la predicción única. La celda de errores imprime las clases peores y la fracción de confusiones dentro del bloque pop/rock.

Esa lectura cierra la pregunta de la asignación: qué modelo generaliza en este corte, y por qué la elección se sostiene cuando la diferencia entre candidatos es más chica que la variación entre artistas. El número de test describe ese pipeline. No autoriza a volver, cambiar la representación y predecir otra vez.
